In [1]:
# We'll fix the script by (1) auto-detecting the model/level/success_rate 
# columns in your CSV (since your file has no headers), and (2) generating 
# per-level figures while printing useful summaries. We'll save the charts
# to /mnt/data/plots and print the file paths so you can download them.
#
# NOTE: To keep compatibility with this environment's plotting policy,
# we generate one chart per figure (no subplots) and use default colors.

import os, csv, re, math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator
from collections import defaultdict

CSV_PATH = "statistics_data_cross-play-data.csv"
OUT_DIR = "plots"
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------- Original helpers (lightly adapted) -----------------
_size_regex = re.compile(r'(\d+(?:\.\d+)?)\s*[Bb]\b')

def extract_size_token(label):
    toks = label.split("-")
    for tok in reversed(toks):
        tok = tok.strip()
        if tok and tok[-1] in "Bb":
            m = _size_regex.fullmatch(tok) or _size_regex.search(tok)
            if m:
                return f"{m.group(1)}B"
    m = _size_regex.search(label)
    if m:
        return f"{m.group(1)}B"
    return None

def alias_model(label, use_short_aliases=True):
    if not use_short_aliases:
        return label
    s = label.lower()
    size = extract_size_token(label)  # '12B' or '1.7B' etc.
    size_num = size[:-1] if size else None
    family = None
    if "gemma" in s:
        family = "gemma"
    elif "nemotron" in s:
        family = "nemotron"
    elif "qwen" in s:
        family = "qwen"
    elif "llama" in s:
        family = "llama"
    elif "mistral" in s and "mixtral" not in s:
        family = "mistral"
    elif "mixtral" in s:
        family = "mixtral"
    elif "phi" in s:
        family = "phi"
    if family and size_num:
        return f"{family}{size_num}"
    base = re.sub(r'^[^_]*_', '', s)  # drop vendor_ prefix if present
    base = base.replace("instruct", "").replace("it", "").replace("chat", "")
    base = re.sub(r'[^a-z0-9.-]+', '', base)  # compact
    if size_num:
        return f"{base.split('-')[0]}{size_num}"
    return base[:20] if len(base) > 20 else base

def parse_pair(model_str):
    parts = model_str.split("_", 3)  # cap at 4 pieces; last piece may contain underscores
    if len(parts) >= 4:
        a = f"{parts[0]}_{parts[1]}"
        b = f"{parts[2]}_{parts[3]}"
        return a, b
    parts_full = model_str.split("_")
    if len(parts_full) == 4:
        return f"{parts_full[0]}_{parts_full[1]}", f"{parts_full[2]}_{parts_full[3]}"
    if len(parts_full) >= 2:
        mid = len(parts_full) // 2
        return "_".join(parts_full[:mid]), "_".join(parts_full[mid:])
    return model_str, model_str

def sem(x):
    x = np.asarray(x, dtype=float)
    n = x.size
    if n <= 1:
        return float("nan")
    return float(np.std(x, ddof=1) / np.sqrt(n))

def alias_sort_key(alias):
    m = re.match(r'([a-zA-Z]+)([\d.]+)?', alias)
    fam = (m.group(1).lower() if m else alias.lower())
    try:
        sz = float(m.group(2)) if (m and m.group(2) is not None) else float("inf")
    except Exception:
        sz = float("inf")
    return (fam, sz, alias.lower())

def polish_axes(ax):
    ax.grid(True, axis="y", linestyle=":", linewidth=0.8, alpha=0.6)
    ax.set_axisbelow(True)
    ax.set_facecolor("#FCFCFD")
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)

def format_as_percent_if_applicable(ax, ymax_est):
    if np.isfinite(ymax_est) and ymax_est <= 1.05:
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v, pos: f"{v*100:.0f}%"))
        ax.set_ylim(0, max(1.0, ymax_est))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=8))

# ----------------- NEW: Auto-detect columns -----------------
def is_intlike(s):
    return re.fullmatch(r'-?\d+', s or '') is not None

def is_floatlike(s):
    return re.fullmatch(r'-?\d+(?:\.\d+)?', s or '') is not None

def looks_like_model_token(s):
    # Heuristic: contains at least 3 underscores and a vendor_model_vendor_model pattern
    return (s is not None and s.count("_") >= 3 and re.search(r'[A-Za-z]+_[A-Za-z0-9-]+_[A-Za-z]+_[A-Za-z0-9-]+', s) is not None)

def detect_columns(rows):
    # rows: list[list[str]], no header
    ncols = max(len(r) for r in rows)
    # pad rows
    rows = [r + ['']*(ncols-len(r)) for r in rows]

    # Model column: maximize 'looks_like_model_token' frequency
    model_scores = []
    for j in range(ncols):
        score = sum(looks_like_model_token(r[j]) for r in rows)
        model_scores.append((score, j))
    idx_model = max(model_scores)[1]

    # Level column: integer-like, small set of unique values (<=10)
    idx_level = None
    for j in range(ncols):
        if j == idx_model: 
            continue
        vals = [r[j] for r in rows if r[j] != '']
        if vals and all(is_intlike(v) for v in vals):
            uniq = set(vals)
            if 1 <= len(uniq) <= 10:
                idx_level = j
                break
    if idx_level is None:
        # fallback: pick the column with most intlike values
        int_counts = []
        for j in range(ncols):
            if j == idx_model: continue
            count = sum(is_intlike(r[j]) for r in rows)
            int_counts.append((count, j))
        idx_level = max(int_counts)[1]

    # Success-rate column: choose rightmost column where most values are floats in [0,1]
    sr_candidates = []
    for j in range(ncols):
        if j in (idx_model, idx_level): 
            continue
        vals = [r[j] for r in rows if r[j] != '']
        if not vals: 
            continue
        numeric = []
        in01 = 0
        for v in vals:
            if is_floatlike(v):
                numeric.append(float(v))
                if 0.0 <= float(v) <= 1.0:
                    in01 += 1
        if numeric:
            frac_in01 = in01 / len(vals)
            # Penalize columns that are constant or near-constant zeros
            uniq = len(set(f"{x:.6g}" for x in numeric))
            score = (frac_in01, uniq, j)  # later columns win by larger j in tie-break
            sr_candidates.append((score, j))
    if sr_candidates:
        idx_sr = max(sr_candidates)[1]
    else:
        # fallback to last column
        idx_sr = ncols - 1

    return idx_model, idx_level, idx_sr

# Load all rows (no headers)
with open(CSV_PATH, newline="", encoding="utf-8") as f:
    rows = list(csv.reader(f))

idx_model, idx_level, idx_sr = detect_columns(rows)

print("Detected columns:")
print(f"  model index      = {idx_model}")
print(f"  level index      = {idx_level}")
print(f"  success_rate idx = {idx_sr}")
print("Sample first 3 rows (model, level, success_rate):")
for r in rows[:3]:
    m = r[idx_model] if idx_model < len(r) else ""
    lvl = r[idx_level] if idx_level < len(r) else ""
    sr = r[idx_sr] if idx_sr < len(r) else ""
    print("   ", m, "|", lvl, "|", sr)

# ----------------- Build aggregates -----------------
USE_SHORT_ALIASES = True
ANNOTATE_COUNTS = True

buckets_by_model = defaultdict(list)  # key=(level, model_alias, "self"/"cross") -> [sr]
cross_matrix = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
chef_models_by_level = defaultdict(set)
assistant_models_by_level = defaultdict(set)
levels_seen = set()
models_seen_by_level = defaultdict(set)

# Parse rows into tuples
bad_rows = 0
for row in rows:
    try:
        model_str = row[idx_model]
        level = row[idx_level]
        sr_raw = row[idx_sr]
        if sr_raw == "":
            continue
        sr = float(sr_raw)
    except Exception:
        bad_rows += 1
        continue
    if not level or not model_str:
        continue

    a_full, b_full = parse_pair(model_str)   # agent_0 (Chef), agent_1 (Assistant)
    a = alias_model(a_full, use_short_aliases=USE_SHORT_ALIASES)
    b = alias_model(b_full, use_short_aliases=USE_SHORT_ALIASES)
    if not a or not b:
        continue

    levels_seen.add(level)
    models_seen_by_level[level].update([a, b])

    if a == b:
        buckets_by_model[(level, a, "self")].append(sr)
    else:
        buckets_by_model[(level, a, "cross")].append(sr)
        buckets_by_model[(level, b, "cross")].append(sr)
        cross_matrix[level][a][b].append(sr)  # Chef=a, Assistant=b
        chef_models_by_level[level].add(a)
        assistant_models_by_level[level].add(b)

print(f"\nParsed {len(rows)} rows; skipped {bad_rows} problematic rows.")
print(f"Levels found: {sorted(levels_seen)}")

def level_key(x):
    try:
        return (0, float(x))
    except Exception:
        return (1, str(x))

levels_sorted = sorted(levels_seen, key=level_key)

def collect_mean_sem_n(vals):
    vals = np.asarray(vals, dtype=float)
    n = int(vals.size)
    if n == 0:
        return float("nan"), float("nan"), 0
    return float(np.mean(vals)), sem(vals), n

def plot_grouped_bars(ax, x_labels, series_keys, mean_map, sem_map, n_map, label_prefix):
    x = np.arange(len(x_labels), dtype=float)
    k = max(1, len(series_keys))
    width = min(0.85 / k, 0.28)

    ymax_est = 0.0
    for i, sk in enumerate(series_keys):
        heights = [mean_map[sk].get(lbl, float("nan")) for lbl in x_labels]
        errors  = [sem_map[sk].get(lbl,  float("nan")) for lbl in x_labels]
        counts  = [n_map[sk].get(lbl,    0)             for lbl in x_labels]
        pos = x + (i - (k - 1) / 2.0) * width

        bars = ax.bar(pos, heights, width, yerr=errors, label=f"{label_prefix} {sk}")

        ymin, ymax = ax.get_ylim()
        yrange = ymax - ymin if ymax > ymin else 1.0
        for rect, h, e, n in zip(bars, heights, errors, counts):
            if not np.isfinite(h):
                continue
            ymax_est = max(ymax_est, float(h) + (float(e) if np.isfinite(e) else 0.0))
            if ANNOTATE_COUNTS:
                ax.text(rect.get_x() + rect.get_width()/2.0,
                        rect.get_height() + 0.02 * yrange,
                        f"n={n}", ha="center", va="bottom", fontsize=9)

    ax.set_xticks(x, x_labels, rotation=60, ha="right")
    return ymax_est

# ----------------- Generate per-level figures -----------------
saved_files = []

for lvl in levels_sorted:
    models_sorted = sorted(models_seen_by_level[lvl], key=alias_sort_key)

    means_self_all, sems_self_all, n_self_all = [], [], []
    means_cross_all, sems_cross_all, n_cross_all = [], [], []
    for m in models_sorted:
        xs = np.array(buckets_by_model.get((lvl, m, "self"), []), dtype=float)
        xc = np.array(buckets_by_model.get((lvl, m, "cross"), []), dtype=float)
        m_s, s_s, n_s = collect_mean_sem_n(xs)
        m_c, s_c, n_c = collect_mean_sem_n(xc)
        means_self_all.append(m_s); sems_self_all.append(s_s); n_self_all.append(n_s)
        means_cross_all.append(m_c); sems_cross_all.append(s_c); n_cross_all.append(n_c)

    chef_models = sorted(chef_models_by_level[lvl], key=alias_sort_key)
    assistant_models = sorted(assistant_models_by_level[lvl], key=alias_sort_key)

    # -------- Panel 1: Self vs Cross by model --------
    fig1, ax1 = plt.subplots(figsize=(max(8, 0.35*len(models_sorted) + 5), 5))
    x = np.arange(len(models_sorted))
    width = 0.42

    # Estimate ymax for formatting
    a_vals = np.nan_to_num(np.array(means_self_all),  nan=0.0) + np.nan_to_num(np.array(sems_self_all),  nan=0.0)
    b_vals = np.nan_to_num(np.array(means_cross_all), nan=0.0) + np.nan_to_num(np.array(sems_cross_all), nan=0.0)
    ymax_est1 = float(max(np.max(a_vals) if a_vals.size else 0.0, np.max(b_vals) if b_vals.size else 0.0))

    b1 = ax1.bar(x - width/2, means_self_all,  width, yerr=sems_self_all,  label="Self-play")
    b2 = ax1.bar(x + width/2, means_cross_all, width, yerr=sems_cross_all, label="Cross-play")

    ax1.set_xticks(x, models_sorted, rotation=60, ha="right")
    ax1.set_xlabel("Model")
    ax1.set_ylabel("Mean success_rate")
    ax1.set_title(f"Level {lvl} — Self vs Cross by model (mean ± SEM)")
    polish_axes(ax1)
    format_as_percent_if_applicable(ax1, ymax_est1)
    ax1.legend(frameon=False, ncol=2)

    ymin, ymax = ax1.get_ylim(); yrange = ymax - ymin
    for rect, n in zip(b1, n_self_all):
        h = rect.get_height()
        if np.isfinite(h):
            ax1.text(rect.get_x()+rect.get_width()/2, h + 0.02*yrange, f"n={n}", ha="center", va="bottom", fontsize=9)
    for rect, n in zip(b2, n_cross_all):
        h = rect.get_height()
        if np.isfinite(h):
            ax1.text(rect.get_x()+rect.get_width()/2, h + 0.02*yrange, f"n={n}", ha="center", va="bottom", fontsize=9)

    p1 = os.path.join(OUT_DIR, f"level_{re.sub(r'[^A-Za-z0-9._-]+','_',str(lvl))}__panel1_self_vs_cross.png")
    fig1.savefig(p1, bbox_inches="tight"); plt.close(fig1); saved_files.append(p1)

    # -------- Panel 2: Cross-play of Chefs --------
    # Build mean/sem/n maps
    means_Chef, sems_Chef, ns_Chef = {}, {}, {}
    for cm in chef_models:
        means_Chef[cm], sems_Chef[cm], ns_Chef[cm] = {}, {}, {}
        for am in assistant_models:
            vals = cross_matrix[lvl][cm].get(am, [])
            m, s, n = collect_mean_sem_n(vals)
            means_Chef[cm][am] = m
            sems_Chef[cm][am] = s
            ns_Chef[cm][am] = n

    fig2, ax2 = plt.subplots(figsize=(max(8, 0.35*len(assistant_models) + 5), 5))
    ymax_est2 = plot_grouped_bars(ax2,
                                  x_labels=assistant_models,
                                  series_keys=chef_models,
                                  mean_map=means_Chef,
                                  sem_map=sems_Chef,
                                  n_map=ns_Chef,
                                  label_prefix="Chef")
    ax2.set_xlabel("Assistant model")
    ax2.set_ylabel("Mean success_rate")
    ax2.set_title(f"Level {lvl} — Cross-play of Chefs (by model)")
    polish_axes(ax2)
    format_as_percent_if_applicable(ax2, ymax_est2 if np.isfinite(ymax_est2) else np.nan)
    ax2.legend(frameon=False, ncol=2)
    p2 = os.path.join(OUT_DIR, f"level_{re.sub(r'[^A-Za-z0-9._-]+','_',str(lvl))}__panel2_cross_by_chefs.png")
    fig2.savefig(p2, bbox_inches="tight"); plt.close(fig2); saved_files.append(p2)

    # -------- Panel 3: Cross-play of Assistants --------
    means_Asst, sems_Asst, ns_Asst = {}, {}, {}
    for am in assistant_models:
        means_Asst[am], sems_Asst[am], ns_Asst[am] = {}, {}, {}
        for cm in chef_models:
            vals = cross_matrix[lvl][cm].get(am, [])
            m, s, n = collect_mean_sem_n(vals)
            means_Asst[am][cm] = m
            sems_Asst[am][cm] = s
            ns_Asst[am][cm] = n

    fig3, ax3 = plt.subplots(figsize=(max(8, 0.35*len(chef_models) + 5), 5))
    ymax_est3 = plot_grouped_bars(ax3,
                                  x_labels=chef_models,
                                  series_keys=assistant_models,
                                  mean_map=means_Asst,
                                  sem_map=sems_Asst,
                                  n_map=ns_Asst,
                                  label_prefix="Assistant")
    ax3.set_xlabel("Chef model")
    ax3.set_ylabel("Mean success_rate")
    ax3.set_title(f"Level {lvl} — Cross-play of Assistants (by model)")
    polish_axes(ax3)
    format_as_percent_if_applicable(ax3, ymax_est3 if np.isfinite(ymax_est3) else np.nan)
    ax3.legend(frameon=False, ncol=2)
    p3 = os.path.join(OUT_DIR, f"level_{re.sub(r'[^A-Za-z0-9._-]+','_',str(lvl))}__panel3_cross_by_assistants.png")
    fig3.savefig(p3, bbox_inches="tight"); plt.close(fig3); saved_files.append(p3)

    # Verbose summaries
    print(f"[Level {lvl}] models: {models_sorted}")
    print(f"  Chef models: {chef_models}")
    print(f"  Assistant models: {assistant_models}")

print("\nSaved figures:")
for p in saved_files:
    print("  ", p)


Detected columns:
  model index      = 0
  level index      = 2
  success_rate idx = 21
Sample first 3 rows (model, level, success_rate):
    Qwen_Qwen3-1.7B_Qwen_Qwen3-8B | 2 | 0.040079590676520754
    Qwen_Qwen3-1.7B_Qwen_Qwen3-8B | 2 | 0.03483606557377049
    Qwen_Qwen3-1.7B_Qwen_Qwen3-8B | 2 | 0.025261324041811847

Parsed 227 rows; skipped 8 problematic rows.
Levels found: ['2']
[Level 2] models: ['gemma4', 'gemma12', 'nemotron14', 'qwen1.7', 'qwen4', 'qwen8', 'qwen14', 'qwen32']
  Chef models: ['gemma4', 'gemma12', 'nemotron14', 'qwen1.7', 'qwen4', 'qwen8', 'qwen14', 'qwen32']
  Assistant models: ['gemma4', 'gemma12', 'nemotron14', 'qwen1.7', 'qwen4', 'qwen8', 'qwen14', 'qwen32']

Saved figures:
   plots/level_2__panel1_self_vs_cross.png
   plots/level_2__panel2_cross_by_chefs.png
   plots/level_2__panel3_cross_by_assistants.png
